In [1]:
#The jupyter notebook can be downloaded using the link at top right corner of the kaggle webpage if not fully rendered in the webpage. 

import numpy as np
import gensim
import os
import re

from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim import corpora

from gensim.models.ldamulticore import LdaMulticore

import pandas as pd

In [2]:
!pip install scispacy

     |████████████████████████████████| 3.0 MB 7.1 MB/s 
     |████████████████████████████████| 12.9 MB 27.4 MB/s 
     |████████████████████████████████| 6.0 MB 31.9 MB/s 
     |████████████████████████████████| 46 kB 3.2 MB/s 
  Created wheel for scispacy: filename=scispacy-0.2.4-py3-none-any.whl size=35203 sha256=8534cf31bc96020c2d3b4c559bd0c9ea283983a5c151c6a7c6922cc0784e94a7
  Stored in directory: /root/.cache/pip/wheels/80/01/69/37a2ab4f9b61773c187d83257ffb31365a5ad57e7779ae5e92
Successfully built scispacy
ERROR: allennlp 0.9.0 has requirement spacy<2.2,>=2.1.0, but you'll have spacy 2.2.3 which is incompatible.
  Attempting uninstall: botocore
    Found existing installation: botocore 1.15.13
    Uninstalling botocore-1.15.13:
      Successfully uninstalled botocore-1.15.13
  Attempting uninstall: rsa
    Found existing installation: rsa 4.0
    Uninstalling rsa-4.0:
      Successfully uninstalled rsa-4.0


In [3]:
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.2.4/en_core_sci_md-0.2.4.tar.gz

     |████████████████████████████████| 70.0 MB 48 kB/s 
  Created wheel for en-core-sci-md: filename=en_core_sci_md-0.2.4-py3-none-any.whl size=70498245 sha256=6f792a8a18637df595e4bf42a693ea7f734327792730f990617c87b1a8a11c64
  Stored in directory: /root/.cache/pip/wheels/0b/d7/91/ad4d42842bdd001ccd505e492eba644c4c97b1fc5558c1d5ee
Successfully built en-core-sci-md


In [4]:
#df = pd.read_csv('metadata.csv')
bucket = 'coviddata'
file = 'metadata.csv'
gcs_url = 'https://%(bucket)s.storage.googleapis.com/%(file)s' % {'bucket':bucket, 'file':file}
df = pd.read_csv(gcs_url)


In [5]:
df.head()

,sha,source_x,title,doi,pmcid,pubmed_id,license,abstract,publish_time,authors,journal,Microsoft Academic Paper ID,WHO #Covidence,has_full_text,full_text_file
0,NaN,Elsevier,Intrauterine virus infections and congenital h...,10.1016/0002-8703(72)90077-4,NaN,4361535.0,els-covid,Abstract The etiologic basis for the vast majo...,1972-12-31,"Overall, James C.",American Heart Journal,NaN,NaN,False,custom_license
1,NaN,Elsevier,Coronaviruses in Balkan nephritis,10.1016/0002-8703(80)90355-5,NaN,6243850.0,els-covid,NaN,1980-03-31,"Georgescu, Leonida; Diosi, Peter; Buţiu, Ioan;...",American Heart Journal,NaN,NaN,False,custom_license
2,NaN,Elsevier,Cigarette smoking and coronary heart disease: ...,10.1016/0002-8703(80)90356-7,NaN,7355701.0,els-covid,NaN,1980-03-31,"Friedman, Gary D",American Heart Journal,NaN,NaN,False,custom_license
3,aecbc613ebdab36753235197ffb4f35734b5ca63,Elsevier,Clinical and immunologic studies in identical ...,10.1016/0002-9343(73)90176-9,NaN,4579077.0,els-covid,"Abstract Middle-aged female identical twins, o...",1973-08-31,"Brunner, Carolyn M.; Horwitz, David A.; Shann,...",The American Journal of Medicine,NaN,NaN,True,custom_license
4,NaN,Elsevier,Epidemiology of community-acquired respiratory...,10.1016/0002-9343(85)90361-4,NaN,4014285.0,els-covid,Abstract Upper respiratory tract infections ar...,1985-06-28,"Garibaldi, Richard A.",The American Journal of Medicine,NaN,NaN,False,custom_license


In [6]:
df2 = df.drop(columns = ['sha', 'source_x', 'pmcid', 'license', 'Microsoft Academic Paper ID', 'WHO #Covidence', 'has_full_text'])

In [7]:
df2.head()

,title,doi,pubmed_id,abstract,publish_time,authors,journal,full_text_file
0,Intrauterine virus infections and congenital h...,10.1016/0002-8703(72)90077-4,4361535.0,Abstract The etiologic basis for the vast majo...,1972-12-31,"Overall, James C.",American Heart Journal,custom_license
1,Coronaviruses in Balkan nephritis,10.1016/0002-8703(80)90355-5,6243850.0,NaN,1980-03-31,"Georgescu, Leonida; Diosi, Peter; Buţiu, Ioan;...",American Heart Journal,custom_license
2,Cigarette smoking and coronary heart disease: ...,10.1016/0002-8703(80)90356-7,7355701.0,NaN,1980-03-31,"Friedman, Gary D",American Heart Journal,custom_license
3,Clinical and immunologic studies in identical ...,10.1016/0002-9343(73)90176-9,4579077.0,"Abstract Middle-aged female identical twins, o...",1973-08-31,"Brunner, Carolyn M.; Horwitz, David A.; Shann,...",The American Journal of Medicine,custom_license
4,Epidemiology of community-acquired respiratory...,10.1016/0002-9343(85)90361-4,4014285.0,Abstract Upper respiratory tract infections ar...,1985-06-28,"Garibaldi, Richard A.",The American Journal of Medicine,custom_license


In [8]:
df2.shape

(44220, 8)

In [9]:
df3 = df2.dropna(subset=['abstract'])

In [10]:
df3.shape

(35806, 8)

In [11]:
df3.head()

,title,doi,pubmed_id,abstract,publish_time,authors,journal,full_text_file
0,Intrauterine virus infections and congenital h...,10.1016/0002-8703(72)90077-4,4361535.0,Abstract The etiologic basis for the vast majo...,1972-12-31,"Overall, James C.",American Heart Journal,custom_license
3,Clinical and immunologic studies in identical ...,10.1016/0002-9343(73)90176-9,4579077.0,"Abstract Middle-aged female identical twins, o...",1973-08-31,"Brunner, Carolyn M.; Horwitz, David A.; Shann,...",The American Journal of Medicine,custom_license
4,Epidemiology of community-acquired respiratory...,10.1016/0002-9343(85)90361-4,4014285.0,Abstract Upper respiratory tract infections ar...,1985-06-28,"Garibaldi, Richard A.",The American Journal of Medicine,custom_license
5,Infectious diarrhea: Pathogenesis and risk fac...,10.1016/0002-9343(85)90367-5,2861742.0,Abstract Our understanding of the pathogenesis...,1985-06-28,"Cantey, J.Robert",The American Journal of Medicine,custom_license
6,New perspectives on the pathogenesis of rheuma...,10.1016/0002-9343(88)90356-7,3052052.0,Abstract In the pathogenesis of rheumatoid art...,1988-10-14,"Zvaifler, Nathan J.",The American Journal of Medicine,custom_license


In [12]:
import en_core_sci_md
nlp = en_core_sci_md.load(disable=["tagger", "parser", "ner"])
nlp.max_length = 2000000

In [13]:
import spacy

In [14]:
from spacy.tokenizer import Tokenizer

In [15]:
def tokenize(doc):
    
    return [token.text for token in nlp(doc) if not token.is_stop and not token.is_punct and not token.pos == 'PRON']

In [16]:
data = df3['abstract'].apply(tokenize)

In [17]:
data

0        [Abstract, etiologic, basis, vast, majority, c...
3        [Abstract, Middle-aged, female, identical, twi...
4        [Abstract, Upper, respiratory, tract, infectio...
5        [Abstract, understanding, pathogenesis, infect...
6        [Abstract, pathogenesis, rheumatoid, arthritis...
                               ...                        
44215    [study, aimed, identify, broad, spectrum, resp...
44216    [Abstract, enter, cells, enveloped, viruses, u...
44217    [Human, parainfluenza, viruses, cause, large, ...
44218    [Abstract, arenaviruses, chiefly, Lassa, virus...
44219    [vigorous, alloimmunity, allograft, usually, r...
Name: abstract, Length: 35806, dtype: object

In [18]:
vect = [nlp(doc).vector for doc in df3['abstract']]

In [19]:
from sklearn.neighbors import NearestNeighbors

In [20]:
nn = NearestNeighbors(n_neighbors=25, algorithm='ball_tree')
nn.fit(vect)

NearestNeighbors(algorithm='ball_tree', leaf_size=30, metric='minkowski',
                 metric_params=None, n_jobs=None, n_neighbors=25, p=2,
                 radius=1.0)

In [21]:
query = "chloroquine hydroxycholoroquine HCoV-19 SARS-CoV-2 coronavirus covid-19 treatment"

In [22]:
query_vect = nlp(query).vector

In [23]:
#find 10 most similar abstracts as the above query
similar_abstracts = nn.kneighbors([query_vect])[1]

In [24]:
for abstract in similar_abstracts:
    print(df3['abstract'].iloc[abstract])

19051    The recent outbreak of respiratory illness in ...
36904    Antiviral agents are urgently needed to fight ...
7684     Summary SARS coronavirus continues to cause sp...
37532    Human coronavirus (HCoV) strain 229E infection...
28684    Severe acute respiratory syndrome (SARS) is an...
8950     Abstract The activity of inosine-5′-monophosph...
43259    Abstract The currently circulating H3N2 and H1...
31893    Two coronaviruses causing severe respiratory d...
9110     Abstract Originally developed and commercializ...
7683     Summary Background The outbreak of severe acut...
4935     Abstract We report on chloroquine, a 4-amino-q...
30910    Ebola virus infection can cause severe hemorrh...
43637    Abstract Ferret and mink coronaviruses typical...
12869    Abstract The anti-RNA virus activity of polyox...
8887     Abstract A novel swine acute diarrhea syndrome...
42245    Human coronaviruses (CoVs) are enveloped virus...
15282    Abstract Objectives The identification and ana.

In [25]:
output = pd.DataFrame((df3['abstract'].iloc[abstract]))


In [26]:
pd.set_option('display.max_colwidth', 0)
output.head(25)
#Output of the top 25 abstracts matching the query with index numbers

,abstract
19051,"The recent outbreak of respiratory illness in Wuhan, China is caused by a novel coronavirus, named 2019-nCoV, which is genetically close to a bat-derived coronavirus. 2019-nCoV is categorized as beta genus coronavirus, same as the two other strains - severe acute respiratory syndrome coronavirus (SARS-CoV) and Middle East respiratory syndrome coronavirus (MERS-CoV). Antiviral drugs commonly used in clinical practice, including neuraminidase inhibitors (oseltamivir, paramivir, zanamivir, etc.), ganciclovir, acyclovir and ribavirin, are invalid for 2019-nCoV and not recommended. Drugs are possibly effective for 2019-nCoV include: remdesivir, lopinavir / ritonavir, lopinavir / ritonavir combined with interferon-β, convalescent plasma, and monoclonal antibodies. But the efficacy and safety of these drugs for 2019-nCoV pneumonia patients need to be assessed by further clinical trials."
36904,"Antiviral agents are urgently needed to fight severe acute respiratory syndrome (SARS). We showed that niclosamide, an existing antihelminthic drug, was able to inhibit replication of a newly discovered coronavirus, SARS-CoV; viral antigen synthesis was totally abolished at a niclosamide concentration of 1.56 μM, as revealed by immunoblot analysis. Thus, niclosamide represents a promising drug candidate for the effective treatment of SARS-CoV infection."
7684,"Summary SARS coronavirus continues to cause sporadic cases of severe acute respiratory syndrome (SARS) in China. No active or passive immunoprophylaxis for disease induced by SARS coronavirus is available. We investigated prophylaxis of SARS coronavirus infection with a neutralising human monoclonal antibody in ferrets, which can be readily infected with the virus. Prophylactic administration of the monoclonal antibody at 10 mg/kg reduced replication of SARS coronavirus in the lungs of infected ferrets by 3·3 logs (95% Cl 2·6–4·0 logs; p<0·001), completely prevented the development of SARS coronavirus-induced macroscopic lung pathology (p=0·013), and abolished shedding of virus in pharyngeal secretions. The data generated in this animal model show that administration of a human monoclonal antibody might offer a feasible and effective prophylaxis for the control of human SARS coronavirus infection."
37532,"Human coronavirus (HCoV) strain 229E infection, but not HCoV strain OC43 infection, of monocytes/macrophages from healthy donors and patients with multiple sclerosis in remission resulted in increased apoptosis, as measured by DNA changes and annexin V staining. Apoptosis correlated with the differential release of infectious virus. HCoV strain 229E titers were 10(3.5) to 10(6) 50% tissue culture-infective doses (TCID(50))/ml, and HCoV strain OC43 titers were only 10(1.2) to 10(2.7) TCID(50)/ml."
28684,"Severe acute respiratory syndrome (SARS) is an infectious disease caused by a newly identified human coronavirus (SARS-CoV). Currently, no effective drug exists to treat SARS-CoV infection. In this study, we investigated whether a panel of commercially available antiviral drugs exhibit in vitro anti–SARS-CoV activity. A drug-screening assay that scores for virus-induced cytopathic effects on cultured cells was used. Tested were 19 clinically approved compounds from several major antiviral pharmacologic classes: nucleoside analogs, interferons, protease inhibitors, reverse transcriptase inhibitors, and neuraminidase inhibitors. Complete inhibition of cytopathic effects of SARS-CoV in culture was observed for interferon subtypes, β-1b, α-n1, α-n3, and human leukocyte interferon α. These findings support clinical testing of approved interferons for the treatment of SARS."
8950,"Abstract The activity of inosine-5′-monophosphate dehydrogenase (IMPDH) inhibitors, mizoribine and ribavirin, against severe acute respiratory syndrome (SARS)-associated coronavirus (SARS-CoV) was determined by plaque reduction and yield reduction assays. Mizoribine and ribavirin selectively inhibite

In [27]:
#From the above abstracts, abstracts 28684, 8950, 7683 appear relevant to our search for newer treatments.
#Abstract 4935 and 34889 is relevant to chloroquine in treating covid-19. 
#Abstract 18811 is relevant to a monoclonal antibody treatment against covid-19.
#Abstract 30643 is relevant to a new target Abelson tyrosine-protein kinase 2 (Abl2) against covid-19.
#Abstract 43973 is important in discussing various approaches towards developing a vaccine and treatments against covid-19.



In [28]:
#Next step will be to inspect the detailed papers for these abstracts.
#Let us inspect the abstracts first and raw some conclusions.

In [29]:
#pd.set_option('display.max_colwidth', 0)
query1 = output.iloc[ 10, : ]
query1.head()

abstract    Abstract We report on chloroquine, a 4-amino-quinoline, as an effective inhibitor of the replication of the severe acute respiratory syndrome coronavirus (SARS-CoV) in vitro. Chloroquine is a clinically approved drug effective against malaria. We tested chloroquine phosphate for its antiviral potential against SARS-CoV-induced cytopathicity in Vero E6 cell culture. Results indicate that the IC50 of chloroquine for antiviral activity (8.8±1.2μM) was significantly lower than its cytostatic activity; CC50 (261.3±14.5μM), yielding a selectivity index of 30. The IC50 of chloroquine for inhibition of SARS-CoV in vitro approximates the plasma concentrations of chloroquine reached during treatment of acute malaria. Addition of chloroquine to infected cultures could be delayed for up to 5h postinfection, without an important drop in antiviral activity. Chloroquine, an old antimalarial drug, may be considered for immediate use in the prevention and treatment of SARS-CoV infections.
N

Abstract 4935: Abstract We report on chloroquine, a 4-amino-quinoline, as an effective inhibitor of the replication of the severe acute respiratory syndrome coronavirus (SARS-CoV) in vitro. Chloroquine is a clinically approved drug effective against malaria. We tested chloroquine phosphate for its antiviral potential against SARS-CoV-induced cytopathicity in Vero E6 cell culture. Results indicate that the IC50 of chloroquine for antiviral activity (8.8±1.2μM) was significantly lower than its cytostatic activity; CC50 (261.3±14.5μM), yielding a selectivity index of 30. The IC50 of chloroquine for inhibition of SARS-CoV in vitro approximates the plasma concentrations of chloroquine reached during treatment of acute malaria. Addition of chloroquine to infected cultures could be delayed for up to 5h postinfection, without an important drop in antiviral activity. Chloroquine, an old antimalarial drug, may be considered for immediate use in the prevention and treatment of SARS-CoV infections.

Abstract 4935 key takeway:
Chloroquine reduced the antiviral activity of SARS-COV (2003 SARS outbreak) in in vitro study and could be an effective treatment for this infection.

In [30]:
query2 = output.iloc[ 19, : ]
query2.head()

abstract    Until recently, human coronaviruses (HCoVs), such as HCoV strain OC43 (HCoV-OC43), were mainly known to cause 15 to 30% of mild upper respiratory tract infections. In recent years, the identification of new HCoVs, including severe acute respiratory syndrome coronavirus, revealed that HCoVs can be highly pathogenic and can cause more severe upper and lower respiratory tract infections, including bronchiolitis and pneumonia. To date, no specific antiviral drugs to prevent or treat HCoV infections are available. We demonstrate that chloroquine, a widely used drug with well-known antimalarial effects, inhibits HCoV-OC43 replication in HRT-18 cells, with a 50% effective concentration (± standard deviation) of 0.306 ± 0.0091 μM and a 50% cytotoxic concentration (± standard deviation) of 419 ± 192.5 μM, resulting in a selectivity index of 1,369. Further, we investigated whether chloroquine could prevent HCoV-OC43-induced death in newborn mice. Our results show that a lethal HCoV-O

Abstract 34889: Until recently, human coronaviruses (HCoVs), such as HCoV strain OC43 (HCoV-OC43), were mainly known to cause 15 to 30% of mild upper respiratory tract infections. In recent years, the identification of new HCoVs, including severe acute respiratory syndrome coronavirus, revealed that HCoVs can be highly pathogenic and can cause more severe upper and lower respiratory tract infections, including bronchiolitis and pneumonia. To date, no specific antiviral drugs to prevent or treat HCoV infections are available. We demonstrate that chloroquine, a widely used drug with well-known antimalarial effects, inhibits HCoV-OC43 replication in HRT-18 cells, with a 50% effective concentration (± standard deviation) of 0.306 ± 0.0091 μM and a 50% cytotoxic concentration (± standard deviation) of 419 ± 192.5 μM, resulting in a selectivity index of 1,369. Further, we investigated whether chloroquine could prevent HCoV-OC43-induced death in newborn mice. Our results show that a lethal HCoV-OC43 infection in newborn C57BL/6 mice can be treated with chloroquine acquired transplacentally or via maternal milk. The highest survival rate (98.6%) of the pups was found when mother mice were treated daily with a concentration of 15 mg of chloroquine per kg of body weight. Survival rates declined in a dose-dependent manner, with 88% survival when treated with 5 mg/kg chloroquine and 13% survival when treated with 1 mg/kg chloroquine. Our results show that chloroquine can be highly effective against HCoV-OC43 infection in newborn mice and may be considered as a future drug against HCoVs.

Note: HCoV strain OC43 (HCoV-OC43) is a different strain than COVID-19 (SARS-COV) but chloroquine seems to have a class effect against coronaviruses. 

****Abstract 43973 is an important review of vaccine and treatment approaches against COVID-19.

In [31]:
#The full text for this abstract can be accessed here : https://github.com/bs3537/DS-Unit-4-Sprint-1-NLP/blob/master/1-s2.0-S2090123220300540-main.pdf

In [32]:
#NLP Topic Modeling for the selected abstract above

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [34]:
vect = TfidfVectorizer(stop_words='english', tokenizer = tokenize, ngram_range=(1,2))

In [35]:
tf = vect.fit_transform(output['abstract'])

/opt/conda/lib/python3.6/site-packages/sklearn/feature_extraction/text.py:385: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['nt'] not in stop_words.
  'stop_words.' % sorted(inconsistent))


In [36]:
from sklearn.decomposition import LatentDirichletAllocation

In [37]:
lda = LatentDirichletAllocation(n_components=50, random_state=0, n_jobs=-1)

In [38]:
lda.fit(tf)

LatentDirichletAllocation(batch_size=128, doc_topic_prior=None,
                          evaluate_every=-1, learning_decay=0.7,
                          learning_method='batch', learning_offset=10.0,
                          max_doc_update_iter=100, max_iter=10,
                          mean_change_tol=0.001, n_components=50, n_jobs=-1,
                          perp_tol=0.1, random_state=0, topic_word_prior=None,
                          total_samples=1000000.0, verbose=0)

In [39]:
def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        message = "\nTopic #%d: " % topic_idx
        message += " ".join([feature_names[i]
                             for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)
    print()

In [40]:
tfidf_feature_names = vect.get_feature_names()
top_words = print_top_words(lda, tfidf_feature_names, 25)
top_words


Topic #0: ≥37.5 ° drugs streamlines e6 cell e6 dynamics simulations dynamics duration clinical duration drugs treatments drugs treatment drugs titanium drugs tested drugs target drugs prevent drugs active drugs possibly drugs m2 drugs inhibitors drugs identified drugs exhibit drugs evaluated drugs cytomegalovirus drugs commonly drugs characterized drugs called

Topic #1: influenza h5n1 seasonal seasonal influenza zanamivir limited avian influenza avian licensed oseltamivir pneumonia amantadine rimantadine amantadine viral pneumonia inhibitors peramivir viral infections circulating h3n2 remains confined peramivir ribavirin prodrug forms viral dissemination forms zanamivir disease transferred confined respiratory

Topic #2: ≥37.5 ° drugs streamlines e6 cell e6 dynamics simulations dynamics duration clinical duration drugs treatments drugs treatment drugs titanium drugs tested drugs target drugs prevent drugs active drugs possibly drugs m2 drugs inhibitors drugs identified drugs exhibit 

In [41]:
!pip install pyLDAvis

In [42]:
import pyLDAvis.gensim

pyLDAvis.enable_notebook()

In [43]:
data = output['abstract'].apply(tokenize)

In [44]:
id2word = corpora.Dictionary(data)

In [45]:
corpus = [id2word.doc2bow(token) for token in data]

In [46]:
lda2 = LdaMulticore(corpus = corpus,
                   id2word = id2word,
                   random_state = 42,
                   num_topics = 15,
                   passes = 10,
                   workers = 4)

In [47]:
lda2.print_topics()

[(0,
  '0.022*"patients" + 0.017*"active" + 0.017*"viruses" + 0.017*"inhibitors" + 0.017*"drug" + 0.011*"associated" + 0.011*"transcriptase" + 0.011*"analogue" + 0.011*"HIV" + 0.011*"reverse"'),
 (1,
  '0.029*"influenza" + 0.017*"H5N1" + 0.017*"seasonal" + 0.017*"viral" + 0.013*"respiratory" + 0.013*"zanamivir" + 0.013*"limited" + 0.009*"virus" + 0.009*"cause" + 0.009*"infections"'),
 (2,
  '0.001*"viral" + 0.001*"clinical" + 0.001*"human" + 0.001*"respiratory" + 0.001*"COVID-19" + 0.001*"coronavirus" + 0.001*"nitazoxanide" + 0.001*"influenza" + 0.001*"virus" + 0.001*"viruses"'),
 (3,
  '0.036*"protease" + 0.031*"feline" + 0.018*"viral" + 0.018*"inhibitors" + 0.018*"coronavirus" + 0.018*"3CL" + 0.014*"replication" + 0.014*"infectious" + 0.014*"FIP" + 0.014*"coronaviruses"'),
 (4,
  '0.032*"2019-nCoV" + 0.032*"coronavirus" + 0.019*"respiratory" + 0.013*"lopinavir" + 0.013*"ritonavir" + 0.013*"clinical" + 0.013*"drugs" + 0.013*"syndrome" + 0.007*"zanamivir" + 0.007*"oseltamivir"'),
 (5,


In [48]:
import re
words = [re.findall(r'"([^"]*)"',t[1]) for t in lda2.print_topics()]

In [49]:
topics = [' '.join(t[0:10]) for t in words]

In [50]:
for id, t in enumerate(topics): 
    print(f"------ Topic {id} ------")
    print(t, end="\n\n")

------ Topic 0 ------
patients active viruses inhibitors drug associated transcriptase analogue HIV reverse

------ Topic 1 ------
influenza H5N1 seasonal viral respiratory zanamivir limited virus cause infections

------ Topic 2 ------
viral clinical human respiratory COVID-19 coronavirus nitazoxanide influenza virus viruses

------ Topic 3 ------
protease feline viral inhibitors coronavirus 3CL replication infectious FIP coronaviruses

------ Topic 4 ------
2019-nCoV coronavirus respiratory lopinavir ritonavir clinical drugs syndrome zanamivir oseltamivir

------ Topic 5 ------
virus clinical respiratory viral influenza Nitazoxanide vitro nitazoxanide treatment trials

------ Topic 6 ------
chloroquine Ebola teicoplanin respiratory viruses HCoV-OC43 treated ± mice HCoVs

------ Topic 7 ------
SARS coronavirus virus respiratory human monoclonal vaccine antibody SARS-coronavirus shedding

------ Topic 8 ------
virus SARS porcine patients coronavirus syndrome influenza respiratory HCoV 

In [51]:
pyLDAvis.gensim.prepare(lda2, corpus, id2word)

/opt/conda/lib/python3.6/site-packages/pyLDAvis/_prepare.py:257: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  return pd.concat([default_term_info] + list(topic_dfs))


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
12     0.002089 -0.169397  1       1        31.966305
5     -0.083668  0.044363  2       1        13.364851
8     -0.154904  0.061777  3       1        13.149855
6     -0.035466 -0.134638  4       1        10.379828
7     -0.077758 -0.066080  5       1        7.467419 
13     0.152519 -0.025888  6       1        6.006211 
1     -0.015423  0.074886  7       1        5.776896 
3      0.148581  0.014715  8       1        5.201589 
0      0.051762  0.117390  9       1        3.655487 
4     -0.024236  0.001355  10      1        2.735028 
14     0.007099  0.013416  11      1        0.059306 
11     0.008241  0.016625  12      1        0.059306 
10     0.006926  0.017440  13      1        0.059306 
9      0.007346  0.016835  14      1        0.059306 
2      0.006892  0.017201  15      1        0.059306 , topic_info=    Category       Freq            Term      Total  loglift  logprob
22   Default  49.000000  coronavirus     49.000000  30.0000  30.0000
63   Default  25.000000  SARS            25.000000  29.0000  29.0000
7    Default  34.000000  SARS-CoV        34.000000  28.0000  28.0000
322  Default  14.000000  influenza       14.000000  27.0000  27.0000
142  Default  32.000000  virus           32.000000  26.0000  26.0000
..       ...        ...    ...                 ...      ...      ...
419  Topic15  0.001388   broad-spectrum  4.350501  -0.6198  -7.0801 
5    Topic15  0.001386   MERS-CoV        21.316536 -2.2108  -7.0819 
60   Topic15  0.001386   trials          5.164648  -0.7932  -7.0819 
403  Topic15  0.001385   C               2.927794  -0.2264  -7.0827 
89   Topic15  0.001384   treatment       11.558467 -1.5997  -7.0828 

[1008 rows x 6 columns], token_table=      Topic      Freq       Term
term                            
979   4      0.792693  0.0091   
980   4      0.792762  0.306    
750   3      0.773349  101      
982   4      0.978604  15       
179   1      0.869469  19       
...  ..           ...  ..       
61    10     0.336022  zanamivir
1021  4      0.831449  ±        
800   3      0.773347  ×        
92    1      0.338478  μM       
92    4      0.676956  μM       

[878 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[13, 6, 9, 7, 8, 14, 2, 4, 1, 5, 15, 12, 11, 10, 3])

In [52]:
#On hovering over 2019-nCoV, this word is most commonly present in topics 1 and 10. 
#COVID-19 word is most commonly present in topic 1. 